# Module 5: Defense Sweeps

This focused notebook owns the longer malicious-fraction, Krum, and non-IID stress sweeps.


## Stage Goal

Stress-test defense behavior after the fixed-attack comparison is working. These sections rerun multiple FL jobs, so keep individual toggles off unless the compute budget is available.


## 1. Notebook Setup

Load shared helpers and make imports independent of notebook launch directory.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display

cwd = Path.cwd()
MODULE_DIR = cwd if cwd.name == "5_Defensive_FL" else cwd / "5_Defensive_FL"
MODULE4_DIR = MODULE_DIR.parent / "4_Adversarial_FL"
MODULE4_SRC_DIR = MODULE4_DIR / "src"
REPO_ROOT = MODULE_DIR.parent

for path in (REPO_ROOT, MODULE4_DIR, MODULE4_SRC_DIR, MODULE_DIR):
    path_str = str(path.resolve())
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from src.notebook_utils import (
    prepare_context,
    record_config_snapshot,
    completed_rows,
    run_krum_hyperparameter_sweep,
    run_malicious_fraction_sweep,
    run_non_iid_stress,
    validate_artifacts,
)


## 2. Configuration and Sweep Controls

Edit this visible `CONFIG` cell to choose sweep grids and toggles. Expensive sweeps are off by default.


In [ ]:
CONFIG = {'data_config': {'dataset_path': './Data/Imagenette',
                 'dataset_name': 'Imagenette',
                 'non_iid_per': 0,
                 'eval_subset': 'attack_eval',
                 'validation_split': {'enabled': True, 'seed': 42, 'selection_fraction': 0.5}},
 'global_config': {'seed': 42, 'device': 'cuda'},
 'module4_handoff': {'enabled': True,
                     'artifacts_dir': '../4_Adversarial_FL/artifacts',
                     'target_checkpoint': 'module4_v3_target.pt',
                     'surrogate_checkpoint': 'module4_surrogate.pt'},
 'artifacts': {'dir': 'artifacts', 'config_snapshot': 'module5_defense_sweeps_config_used.json'},
 'stage': {'name': 'defense_sweeps', 'notebook': 'defense_sweeps.ipynb'},
 'defense_sweeps': {'run_malicious_fraction_sweep': False,
                    'run_krum_hyperparameter_sweep': False,
                    'run_non_iid_stress': False},
 'model_config': {'module': 'model',
                  'name': 'MobileNetV3Transfer',
                  'kwargs': {'pretrained': False, 'num_classes': 10, 'dropout': 0.1}},
 'algorithms': {'FedAvg': {'fed_config': {'algorithm': 'FedAvg',
                                          'fraction_clients': 0.2,
                                          'num_clients': 50,
                                          'num_rounds': 12,
                                          'num_epochs': 3,
                                          'batch_size': 64,
                                          'global_stepsize': 1.0,
                                          'local_stepsize': 0.002,
                                          'criterion': 'torch.nn.CrossEntropyLoss'},
                           'optim_config': {}}},
 'defense': {'name': 'trimmed_mean', 'trim_fraction': 0.1},
 'attack': {'seed': 42,
            'malicious_fraction': 0.1,
            'malicious_client_selection': {'mode': 'seeded_random', 'client_ids': []},
            'start_round': 3,
            'attack': {'type': 'pgd',
                       'poison_rate': 0.2,
                       'target_label': 0,
                       'epsilon': 0.03137255,
                       'step_size': 0.00784314,
                       'iters': 20,
                       'criterion': 'torch.nn.CrossEntropyLoss',
                       'poison_rate_schedule': {'type': 'linear', 'start': 0.05, 'end': 0.2}},
            'surrogate': {'checkpoint': '../4_Adversarial_FL/artifacts/module4_surrogate.pt',
                          'checkpoint_source': 'train_surrogate.ipynb',
                          'pretrained': False,
                          'finetune_epochs': 0,
                          'local_finetune_epochs': 0,
                          'learning_rate': 0.001,
                          'weight_decay': 0.0,
                          'batch_size': 64,
                          'freeze_backbone': False,
                          'early_stop_patience': 0}},
 'experiments': {'defenses': [{'name': 'fedavg'},
                              {'name': 'clipping', 'clip_norm': 5.0},
                              {'name': 'median'},
                              {'name': 'trimmed_mean', 'trim_fraction': 0.1},
                              {'name': 'krum', 'byzantine_f': 2}],
                 'malicious_fraction_sweep': [0.0, 0.05, 0.1, 0.2, 0.3],
                 'non_iid_sweep': [0.0, 0.5, 0.8],
                 'non_iid_defenses': [{'name': 'fedavg'},
                                      {'name': 'clipping', 'clip_norm': 5.0},
                                      {'name': 'trimmed_mean', 'trim_fraction': 0.1},
                                      {'name': 'krum', 'byzantine_f': 2}],
                 'krum_byzantine_f_sweep': [1, 2, 5]}}

context = prepare_context(CONFIG, stage_name="defense_sweeps")
config_snapshot_path = record_config_snapshot(context)
sweep_controls = context.config.get("defense_sweeps", {})

RUN_MALICIOUS_FRACTION_SWEEP = bool(
    sweep_controls.get("run_malicious_fraction_sweep", False)
)
RUN_KRUM_HYPERPARAMETER_SWEEP = bool(
    sweep_controls.get("run_krum_hyperparameter_sweep", False)
)
RUN_NON_IID_STRESS = bool(
    sweep_controls.get("run_non_iid_stress", False)
)

{
    "malicious_fraction_sweep": RUN_MALICIOUS_FRACTION_SWEEP,
    "krum_hyperparameter_sweep": RUN_KRUM_HYPERPARAMETER_SWEEP,
    "non_iid_stress": RUN_NON_IID_STRESS,
}


## 3. Malicious-Fraction Sweep

Vary the malicious-client fraction while keeping the attack recipe and defense settings fixed.


In [ ]:
malicious_fraction_rows = (
    run_malicious_fraction_sweep(context)
    if RUN_MALICIOUS_FRACTION_SWEEP
    else []
)
if malicious_fraction_rows:
    display(pd.DataFrame(malicious_fraction_rows))
    for plot_name in [
        "module5_malicious_fraction_accuracy.png",
        "module5_malicious_fraction_global_target_label_asr.png",
    ]:
        plot_path = context.artifact_path(plot_name)
        if plot_path.exists():
            display(Image(filename=str(plot_path)))
else:
    print("Malicious-fraction sweep is disabled in CONFIG['defense_sweeps'].")


## 4. Krum Hyperparameter Sweep

Vary `byzantine_f`; infeasible settings are recorded as skipped rows.


In [ ]:
krum_sweep_rows = (
    run_krum_hyperparameter_sweep(context)
    if RUN_KRUM_HYPERPARAMETER_SWEEP
    else []
)
if krum_sweep_rows:
    display(pd.DataFrame(krum_sweep_rows))
    for plot_name in [
        "module5_krum_byzantine_f_accuracy.png",
        "module5_krum_byzantine_f_global_target_label_asr.png",
    ]:
        plot_path = context.artifact_path(plot_name)
        if plot_path.exists():
            display(Image(filename=str(plot_path)))
else:
    print("Krum hyperparameter sweep is disabled in CONFIG['defense_sweeps'].")


## 5. Non-IID Stress Test

Vary honest-client label skew and compare the configured stress-test defenses.


In [ ]:
non_iid_rows = (
    run_non_iid_stress(context)
    if RUN_NON_IID_STRESS
    else []
)
if non_iid_rows:
    display(pd.DataFrame(non_iid_rows))
    for plot_name in [
        "module5_non_iid_accuracy.png",
        "module5_non_iid_global_target_label_asr.png",
    ]:
        plot_path = context.artifact_path(plot_name)
        if plot_path.exists():
            display(Image(filename=str(plot_path)))
else:
    print("Non-IID stress test is disabled in CONFIG['defense_sweeps'].")


## Handoff Artifacts

Validate the artifacts produced by the enabled sweep sections.


In [ ]:
expected_artifacts = [config_snapshot_path]

if malicious_fraction_rows:
    expected_artifacts.append(context.artifact_path("module5_malicious_fraction_sweep.json"))
    if completed_rows(malicious_fraction_rows):
        expected_artifacts.extend([
            context.artifact_path("module5_malicious_fraction_accuracy.png"),
            context.artifact_path("module5_malicious_fraction_global_target_label_asr.png"),
        ])

if krum_sweep_rows:
    expected_artifacts.append(context.artifact_path("module5_krum_byzantine_f_sweep.json"))
    if completed_rows(krum_sweep_rows):
        expected_artifacts.extend([
            context.artifact_path("module5_krum_byzantine_f_accuracy.png"),
            context.artifact_path("module5_krum_byzantine_f_global_target_label_asr.png"),
        ])

if non_iid_rows:
    expected_artifacts.append(context.artifact_path("module5_non_iid_defense_stress.json"))
    if completed_rows(non_iid_rows):
        expected_artifacts.extend([
            context.artifact_path("module5_non_iid_accuracy.png"),
            context.artifact_path("module5_non_iid_global_target_label_asr.png"),
        ])

validate_artifacts(expected_artifacts)
